# Cellpose/Omnipose: Two-Stage Training (One Notebook)

This notebook mirrors the same flow as `train_two_stage_mit_b3.ipynb`:
1. Stage 1 pretrain on non-cups
2. Stage 2 finetune on cups
3. Validation metrics and final visualization


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path('.').resolve()
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
if not VENV_PYTHON.exists():
    raise FileNotFoundError(f'Python from .venv not found: {VENV_PYTHON}')

print('Project root:', PROJECT_ROOT)
print('Python:', VENV_PYTHON)


In [ ]:
# Paths and logs
SPLITS_DIR = Path('data/splits')
LOG_STAGE1 = Path('runs/cellpose_stage1_train.log')
LOG_STAGE2 = Path('runs/cellpose_stage2_train.log')
CP_STAGE1_DIR = Path('experiments/cellpose/stage1')
CP_STAGE2_DIR = Path('experiments/cellpose/stage2')

print('Stage1 log:', LOG_STAGE1)
print('Stage2 log:', LOG_STAGE2)


In [ ]:
# Helper to run shell commands with notebook-friendly progress + realtime curves
import re
from tqdm.auto import tqdm
from IPython.display import display
import matplotlib.pyplot as plt


def run_cmd(
    cmd,
    cwd='.',
    log_path=None,
    epoch_total=None,
    quiet_tqdm_lines=True,
    live_plots=True,
    live_plot_every=1,
):
    print('\n>>>', ' '.join(cmd))

    log_f = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_f = log_path.open('w', encoding='utf-8')
        print('logging to:', log_path)

    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'

    train_pbar = None
    val_pbar = None
    current_epoch = None

    hist_train_loss = []
    hist_val_f1 = []
    plot_handle = None

    train_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[train\]:.*?\|\s*(\d+)/(\d+)\s*\[.*loss=([0-9.]+)")
    val_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[val\]:.*?\|\s*(\d+)/(\d+)\s*\[")
    epoch_summary_re = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    def redraw_curves(final=False):
        nonlocal plot_handle
        if not live_plots or not hist_train_loss:
            return
        if (not final) and (len(hist_train_loss) % max(1, int(live_plot_every)) != 0):
            return

        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(hist_train_loss, label='train_loss')
        ax[0].set_title('Train Loss')
        ax[0].legend()

        ax[1].plot(hist_val_f1, label='val_f1')
        ax[1].set_title('Val F1')
        ax[1].legend()

        if plot_handle is None:
            plot_handle = display(fig, display_id=True)
        else:
            plot_handle.update(fig)
        plt.close(fig)

    def ensure_epoch(epoch_num):
        nonlocal current_epoch, train_pbar, val_pbar
        if current_epoch == epoch_num:
            return

        if train_pbar is not None:
            train_pbar.close()
            train_pbar = None
        if val_pbar is not None:
            val_pbar.close()
            val_pbar = None

        current_epoch = epoch_num

    try:
        proc = subprocess.Popen(
            cmd,
            cwd=str(Path(cwd).resolve()),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )

        for raw in proc.stdout:
            if log_f is not None:
                log_f.write(raw)

            line = raw.rstrip('\n')

            mt = train_re.search(line)
            if mt:
                ep = int(mt.group(1)); ep_tot = int(mt.group(2))
                cur = int(mt.group(3)); tot = int(mt.group(4)); loss = float(mt.group(5))
                ensure_epoch(ep)
                if train_pbar is None or train_pbar.total != tot:
                    if train_pbar is not None:
                        train_pbar.close()
                    train_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [train]')
                if cur >= train_pbar.n:
                    train_pbar.update(cur - train_pbar.n)
                train_pbar.set_postfix(loss=f'{loss:.4f}')
                continue

            mv = val_re.search(line)
            if mv:
                ep = int(mv.group(1)); ep_tot = int(mv.group(2))
                cur = int(mv.group(3)); tot = int(mv.group(4))
                ensure_epoch(ep)
                if val_pbar is None or val_pbar.total != tot:
                    if val_pbar is not None:
                        val_pbar.close()
                    val_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [val]')
                if cur >= val_pbar.n:
                    val_pbar.update(cur - val_pbar.n)
                continue

            me = epoch_summary_re.search(line)
            if me:
                tl = float(me.group(3)); vf1 = float(me.group(4))
                hist_train_loss.append(tl)
                hist_val_f1.append(vf1)
                redraw_curves(final=False)
                print(line)
                continue

            if line.startswith('Epoch time:'):
                print(line)
                continue

            if 'Saved best:' in line or 'Loaded init checkpoint:' in line:
                print(line)
            elif not quiet_tqdm_lines and line:
                print(line)

        proc.wait()
    finally:
        redraw_curves(final=True)
        if train_pbar is not None:
            train_pbar.close()
        if val_pbar is not None:
            val_pbar.close()
        if log_f is not None:
            log_f.close()

    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {cmd}')




In [ ]:
# Step 1: generate/re-generate two-stage ID splits
run_cmd([
    str(VENV_PYTHON),
    'tools/make_two_stage_ids.py',
    '--images_dir', 'trainable_pool/images',
    '--instances_dir', 'trainable_pool/instances',
    '--out_dir', str(SPLITS_DIR),
    '--cups_prefix', 'data_cups',
    '--val_split_stage1', '0.15',
    '--val_split_stage2', '0.15',
    '--seed', '42',
])

# Build cellpose folders: image.png + image_masks.png
# Large masks can trigger ArrayMemoryError in Cellpose flow conversion, so we cap max side.
import cv2
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')
CP_MAX_SIDE = 2048

def _find_img(image_id):
    for ext in IMG_EXTS:
        p = Path('trainable_pool/images') / f'{image_id}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(image_id)

def _read_ids(p):
    return [x.strip() for x in Path(p).read_text(encoding='utf-8').splitlines() if x.strip()]

def _resize_if_needed(img, msk, max_side=CP_MAX_SIDE):
    h, w = img.shape[:2]
    side = max(h, w)
    if side <= max_side:
        return img, msk, False
    scale = max_side / float(side)
    new_w = max(32, int(round(w * scale)))
    new_h = max(32, int(round(h * scale)))
    img_r = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    msk_r = cv2.resize(msk, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    return img_r, msk_r, True

def _mk(ids, out_dir):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    for old in out.glob('*_img.png'):
        old.unlink()
    for old in out.glob('*_masks.png'):
        old.unlink()

    resized = 0
    for image_id in ids:
        img = cv2.imread(str(_find_img(image_id)), cv2.IMREAD_COLOR)
        msk = cv2.imread(str(Path('trainable_pool/instances') / f'{image_id}.png'), cv2.IMREAD_UNCHANGED)
        if img is None or msk is None:
            raise FileNotFoundError(f'Missing image or mask for {image_id}')
        img, msk, was_resized = _resize_if_needed(img, msk)
        resized += int(was_resized)
        cv2.imwrite(str(out / f'{image_id}_img.png'), img)
        cv2.imwrite(str(out / f'{image_id}_masks.png'), msk)
    print(f'{out}: written {len(ids)} pairs, resized {resized}')

s1_tr = _read_ids(SPLITS_DIR / 'stage1_pretrain_train_ids.txt')
s1_va = _read_ids(SPLITS_DIR / 'stage1_pretrain_val_ids.txt')
s2_tr = _read_ids(SPLITS_DIR / 'stage2_finetune_train_ids.txt')
s2_va = _read_ids(SPLITS_DIR / 'stage2_finetune_val_ids.txt')

_mk(s1_tr, CP_STAGE1_DIR / 'train')
_mk(s1_va, CP_STAGE1_DIR / 'val')
_mk(s2_tr, CP_STAGE2_DIR / 'train')
_mk(s2_va, CP_STAGE2_DIR / 'val')
print('Prepared cellpose dirs')



In [ ]:
# Show split sizes
for p in [
    SPLITS_DIR / 'stage1_pretrain_train_ids.txt',
    SPLITS_DIR / 'stage1_pretrain_val_ids.txt',
    SPLITS_DIR / 'stage2_finetune_train_ids.txt',
    SPLITS_DIR / 'stage2_finetune_val_ids.txt',
]:
    n = len([x for x in p.read_text(encoding='utf-8').splitlines() if x.strip()])
    print(f'{p}: {n}')


In [ ]:
# Step 2: Stage 1 training (pretrain on non-cups)
import torch
run_cmd([str(VENV_PYTHON), '-m', 'pip', 'install', '-U', 'cellpose==3.1.1.1', 'tifffile'])

cmd = [
    str(VENV_PYTHON), '-m', 'cellpose',
    '--train',
    '--dir', str(CP_STAGE1_DIR/'train'),
    '--test_dir', str(CP_STAGE1_DIR/'val'),
    '--img_filter', '_img',
    '--mask_filter', '_masks',
    '--chan', '0', '--chan2', '0',
    '--pretrained_model', 'cyto3',
    '--n_epochs', '120',
    '--batch_size', '8',
    '--learning_rate', '0.1',
    '--weight_decay', '1e-5',
    '--save_every', '20',
    '--model_name_out', 'cellpose_stage1',
]
if torch.cuda.is_available():
    cmd.append('--use_gpu')
run_cmd(cmd, log_path=LOG_STAGE1, quiet_tqdm_lines=False, live_plots=True)


In [ ]:
# Verify Stage 1 checkpoint exists
stage1_models = sorted((CP_STAGE1_DIR/'train'/'models').glob('cellpose_stage1*'))
print('stage1 models:', [str(x) for x in stage1_models[-5:]])
if not stage1_models:
    raise FileNotFoundError('No stage1 cellpose model found')
stage1_best = stage1_models[-1]
print('using stage1 model:', stage1_best)


In [ ]:
# Step 3: Stage 2 training (finetune on cups only)
cmd = [
    str(VENV_PYTHON), '-m', 'cellpose',
    '--train',
    '--dir', str(CP_STAGE2_DIR/'train'),
    '--test_dir', str(CP_STAGE2_DIR/'val'),
    '--img_filter', '_img',
    '--mask_filter', '_masks',
    '--chan', '0', '--chan2', '0',
    '--pretrained_model', str(stage1_best),
    '--n_epochs', '120',
    '--batch_size', '8',
    '--learning_rate', '0.05',
    '--weight_decay', '1e-5',
    '--save_every', '20',
    '--model_name_out', 'cellpose_stage2',
]
if torch.cuda.is_available():
    cmd.append('--use_gpu')
run_cmd(cmd, log_path=LOG_STAGE2, quiet_tqdm_lines=False, live_plots=True)


In [ ]:
# Verify Stage 2 checkpoints
stage2_models = sorted((CP_STAGE2_DIR/'train'/'models').glob('cellpose_stage2*'))
print('stage2 models:', [str(x) for x in stage2_models[-5:]])
if not stage2_models:
    raise FileNotFoundError('No stage2 cellpose model found')
stage2_best = stage2_models[-1]
print('using stage2 model:', stage2_best)


In [ ]:
# Training curves from logs
import re
import numpy as np
import matplotlib.pyplot as plt


def parse_train_log(path):
    path = Path(path)
    if not path.exists():
        print(f'log missing: {path}')
        return None

    epoch, train_loss = [], []
    val_f1, val_merge, val_split, val_count = [], [], [], []

    pat = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    for ln in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        m = pat.search(ln)
        if not m:
            continue
        epoch.append(int(m.group(1)))
        train_loss.append(float(m.group(3)))
        val_f1.append(float(m.group(4)))
        val_merge.append(float(m.group(5)))
        val_split.append(float(m.group(6)))
        val_count.append(float(m.group(7)))

    if not epoch:
        print(f'no parsed epoch lines in: {path}')
        return None

    return {
        'epoch': np.array(epoch),
        'train_loss': np.array(train_loss),
        'val_f1': np.array(val_f1),
        'val_merge': np.array(val_merge),
        'val_split': np.array(val_split),
        'val_count_err': np.array(val_count),
        'path': path,
    }


def plot_stage_curves(data, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(data['train_loss'], label='train_loss')
    ax[0].set_title('Train Loss')
    ax[0].legend()

    ax[1].plot(data['val_f1'], label='val_f1')
    ax[1].set_title('Val F1')
    ax[1].legend()

    print(title)
    plt.show()


stage1 = parse_train_log(LOG_STAGE1)
stage2 = parse_train_log(LOG_STAGE2)

if stage1 is not None:
    print('Stage1 epochs:', len(stage1['epoch']), 'best val_f1:', float(stage1['val_f1'].max()))
    plot_stage_curves(stage1, 'Stage 1: Pretrain (non-cups)')

if stage2 is not None:
    print('Stage2 epochs:', len(stage2['epoch']), 'best val_f1:', float(stage2['val_f1'].max()))
    plot_stage_curves(stage2, 'Stage 2: Finetune (cups)')



## Notes

- For Omnipose-like behavior, install `omnipose` and test stronger boundary separation settings.
- Tune `cellprob_threshold`, `flow_threshold`, `min_size` for your cups domain.
- If GPU memory is low, reduce `batch_size`.


In [ ]:
# Final visualization cell
from cellpose import models
import torch, cv2, numpy as np, matplotlib.pyplot as plt

img_path = 'IMG_4377.jpg'
cp_model = models.CellposeModel(gpu=torch.cuda.is_available(), pretrained_model=str(stage2_best))
img = cv2.imread(img_path, cv2.IMREAD_COLOR)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
masks, *_ = cp_model.eval(img_rgb, channels=[0,0], diameter=None, cellprob_threshold=0.0, flow_threshold=0.4, min_size=4)
edges = cv2.Canny((masks>0).astype(np.uint8)*255, 50, 150)>0
vis = img_rgb.copy(); vis[edges] = [0,255,0]

plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.title('Input'); plt.imshow(img_rgb); plt.axis('off')
plt.subplot(1,2,2); plt.title(f'Cellpose instances: {int(np.max(masks))}'); plt.imshow(vis); plt.axis('off')
plt.tight_layout(); plt.show()
